# IMS Inference Pipeline Validation

This notebook validates the deterministic v0.1.0 IMS inference pipeline before any API, backend, database, or frontend integration.

It loads the saved selected artifact and metadata without retraining, runs inference against the same chronological input used by anomaly validation, checks parity with `ims_validated_anomaly_scores.csv`, and writes JSON-safe validation outputs.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.inference.ims_anomaly_inference import ImsAnomalyInferencePipeline

FEATURE_PATH = PROJECT_ROOT / "data" / "processed" / "ims_features.csv"
VALIDATED_PATH = PROJECT_ROOT / "data" / "processed" / "ims_validated_anomaly_scores.csv"
ARTIFACT_PATH = PROJECT_ROOT / "artifacts" / "models" / "ims_selected_anomaly_model_v0_1_0.joblib"
METADATA_PATH = PROJECT_ROOT / "artifacts" / "models" / "ims_selected_anomaly_model_v0_1_0.json"
VALIDATION_JSON_PATH = PROJECT_ROOT / "artifacts" / "metrics" / "ims_inference_pipeline_validation.json"
PREDICTION_JSON_PATH = PROJECT_ROOT / "data" / "processed" / "ims_inference_pipeline_sample_predictions.json"

NUMERIC_TOLERANCE = 1e-6
VALIDATION_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)

ARTIFACT_PATH, METADATA_PATH

(WindowsPath('C:/Users/Balsem/Desktop/GMAO/ai-service/artifacts/models/ims_selected_anomaly_model_v0_1_0.joblib'),
 WindowsPath('C:/Users/Balsem/Desktop/GMAO/ai-service/artifacts/models/ims_selected_anomaly_model_v0_1_0.json'))

## 1. Load Inputs and v0.1.0 Artifact

In [2]:
features = pd.read_csv(FEATURE_PATH, parse_dates=["timestamp"])
validated = pd.read_csv(VALIDATED_PATH, parse_dates=["timestamp"])
pipeline = ImsAnomalyInferencePipeline(ARTIFACT_PATH, METADATA_PATH)

evaluation_start = validated["timestamp"].min()
inference_input = features[
    features["experiment"].eq("1st_test") & features["timestamp"].ge(evaluation_start)
].copy()

artifact_summary = pd.DataFrame(
    [
        {
            "model_version": pipeline.version,
            "selected_method": pipeline.artifact["selected_method"],
            "validated_experiments": ", ".join(sorted(pipeline.validated_experiments)),
            "cross_experiment_validation_performed": pipeline.artifact["cross_experiment_validation_performed"],
            "feature_order": ", ".join(pipeline.feature_order),
        }
    ]
)

artifact_summary

,model_version,selected_method,validated_experiments,cross_experiment_validation_performed,feature_order
0,0.1.0,weighted,1st_test,False,"rms, standard_deviation, peak_to_peak, kurtosi..."


## 2. Batch Inference and Parity with Notebook 04

In [3]:
batch_output = pipeline.predict_batch(inference_input)
merged = batch_output.merge(validated, on=["experiment", "timestamp", "bearing"], suffixes=("_new", "_validated"))

boolean_columns = [
    "z_is_anomaly",
    "if_is_anomaly",
    "or_raw_alert",
    "and_raw_alert",
    "weighted_raw_alert",
    "z_persistent_alert",
    "if_persistent_alert",
    "or_persistent_alert",
    "and_persistent_alert",
    "weighted_persistent_alert",
]
numeric_columns = [
    "z_anomaly_score",
    "if_anomaly_score",
    "z_score_normalized",
    "if_score_normalized",
    "or_score",
    "and_score",
    "weighted_score",
]

boolean_mismatches = {
    column: int((merged[f"{column}_new"] != merged[f"{column}_validated"]).sum())
    for column in boolean_columns
}
numeric_max_abs_diff = {
    column: float(np.abs(merged[f"{column}_new"] - merged[f"{column}_validated"]).max())
    for column in numeric_columns
}

parity_passed = (
    len(batch_output) == len(validated)
    and len(merged) == len(validated)
    and all(value == 0 for value in boolean_mismatches.values())
    and all(value <= NUMERIC_TOLERANCE for value in numeric_max_abs_diff.values())
)
if not parity_passed:
    raise AssertionError("Inference parity with ims_validated_anomaly_scores.csv failed.")

parity_summary = pd.DataFrame(
    [
        {
            "rows": len(batch_output),
            "matched_rows": len(merged),
            "boolean_mismatches": sum(boolean_mismatches.values()),
            "max_numeric_abs_diff": max(numeric_max_abs_diff.values()),
            "numeric_tolerance": NUMERIC_TOLERANCE,
            "parity_passed": parity_passed,
        }
    ]
)

parity_summary

,rows,matched_rows,boolean_mismatches,max_numeric_abs_diff,numeric_tolerance,parity_passed
0,4312,4312,0,4.258554e-07,0.000001,True


## 3. Streaming Equivalence

In [4]:
streaming_timestamps = inference_input["timestamp"].drop_duplicates().head(100)
streaming_input = inference_input[inference_input["timestamp"].isin(streaming_timestamps)].copy()
streaming_pipeline = ImsAnomalyInferencePipeline(ARTIFACT_PATH, METADATA_PATH)
streaming_parts = []
for _, timestamp_frame in streaming_input.groupby("timestamp", sort=True):
    streaming_parts.append(streaming_pipeline.predict_timestamp(timestamp_frame))
streaming_output = pd.concat(streaming_parts, ignore_index=True)
streaming_batch_output = ImsAnomalyInferencePipeline(ARTIFACT_PATH, METADATA_PATH).predict_batch(streaming_input)

pd.testing.assert_frame_equal(streaming_batch_output.reset_index(drop=True), streaming_output.reset_index(drop=True))
streaming_summary = pd.DataFrame(
    [
        {
            "timestamps_checked": len(streaming_timestamps),
            "batch_rows": len(streaming_batch_output),
            "streaming_rows": len(streaming_output),
            "equivalent": True,
        }
    ]
)
streaming_summary

,timestamps_checked,batch_rows,streaming_rows,equivalent
0,100,400,400,True


## 4. JSON-Safe Output Contract

In [5]:
json_records = pipeline.to_json_records(batch_output.head(20))
json.dumps(json_records, allow_nan=False)
PREDICTION_JSON_PATH.write_text(json.dumps(json_records, indent=2, allow_nan=False), encoding="utf-8")

json_records[:3]

[{'modelVersion': '0.1.0',
  'experiment': '1st_test',
  'timestamp': '2003-11-15T18:18:46',
  'bearing': 1,
  'anomalyScore': 0.43199267712606565,
  'riskScore': 43,
  'riskLevel': 'MONITOR',
  'rawAnomaly': False,
  'persistentAlert': False,
  'componentScores': {'zScore': 0.725035812517097,
   'isolationForest': 0.13894954173503424},
  'reasonCodes': ['ELEVATED_ROLLING_DEVIATION'],
  'prototypeResult': True},
 {'modelVersion': '0.1.0',
  'experiment': '1st_test',
  'timestamp': '2003-11-15T18:18:46',
  'bearing': 2,
  'anomalyScore': 0.5087444605237118,
  'riskScore': 51,
  'riskLevel': 'MONITOR',
  'rawAnomaly': False,
  'persistentAlert': False,
  'componentScores': {'zScore': 0.8579037141012718,
   'isolationForest': 0.15958520694615178},
  'reasonCodes': ['ELEVATED_ROLLING_DEVIATION'],
  'prototypeResult': True},
 {'modelVersion': '0.1.0',
  'experiment': '1st_test',
  'timestamp': '2003-11-15T18:18:46',
  'bearing': 3,
  'anomalyScore': 0.5039138239702519,
  'riskScore': 50,
  

## 5. Save Validation Report

In [6]:
validation_report = {
    "modelVersion": pipeline.version,
    "selectedMethod": pipeline.artifact["selected_method"],
    "validatedExperiments": sorted(pipeline.validated_experiments),
    "crossExperimentValidationPerformed": bool(pipeline.artifact["cross_experiment_validation_performed"]),
    "crossExperimentValidationEvidence": pipeline.artifact["cross_experiment_validation_evidence"],
    "featureOrder": pipeline.feature_order,
    "riskLevels": pipeline.risk_levels,
    "parity": {
        "rows": int(len(batch_output)),
        "matchedRows": int(len(merged)),
        "booleanMismatches": boolean_mismatches,
        "numericMaxAbsDiff": numeric_max_abs_diff,
        "numericTolerance": NUMERIC_TOLERANCE,
        "passed": bool(parity_passed),
    },
    "streamingEquivalence": {
        "timestampsChecked": int(len(streaming_timestamps)),
        "batchRows": int(len(streaming_batch_output)),
        "streamingRows": int(len(streaming_output)),
        "passed": True,
    },
    "outputs": {
        "samplePredictions": PREDICTION_JSON_PATH.as_posix(),
    },
    "limitations": [
        "The v0.1.0 artifact is validated only on the later chronological portion of 1st_test.",
        "No cross-experiment validation on 2nd_test or 3rd_test has been performed yet.",
        "Risk levels are stable prototype bands and are not industrially validated thresholds.",
        "No FastAPI, NestJS, MongoDB, or frontend integration is included in this phase.",
    ],
}

VALIDATION_JSON_PATH.write_text(json.dumps(validation_report, indent=2, allow_nan=False), encoding="utf-8")
json.loads(VALIDATION_JSON_PATH.read_text(encoding="utf-8"))
validation_report

{'modelVersion': '0.1.0',
 'selectedMethod': 'weighted',
 'validatedExperiments': ['1st_test'],
 'crossExperimentValidationPerformed': False,
 'crossExperimentValidationEvidence': 'Validation metrics JSON aggregation_summary contains only 1st_test.',
 'featureOrder': ['rms',
  'standard_deviation',
  'peak_to_peak',
  'kurtosis',
  'skewness',
  'crest_factor',
  'spectral_energy',
  'dominant_frequency_hz'],
 'riskLevels': [{'level': 'NORMAL', 'min': 0, 'max': 39},
  {'level': 'MONITOR', 'min': 40, 'max': 69},
  {'level': 'HIGH', 'min': 70, 'max': 84},
  {'level': 'CRITICAL', 'min': 85, 'max': 100}],
 'parity': {'rows': 4312,
  'matchedRows': 4312,
  'booleanMismatches': {'z_is_anomaly': 0,
   'if_is_anomaly': 0,
   'or_raw_alert': 0,
   'and_raw_alert': 0,
   'weighted_raw_alert': 0,
   'z_persistent_alert': 0,
   'if_persistent_alert': 0,
   'or_persistent_alert': 0,
   'and_persistent_alert': 0,
   'weighted_persistent_alert': 0},
  'numericMaxAbsDiff': {'z_anomaly_score': 4.258554